# GPS Data Quality — Per .raw File

Notebook 09 assessed GPS quality grouped by download folder, but those folders are
arbitrary batches.  This notebook repeats the key visualizations grouped by individual
`.raw` files, which are the natural recording units — each file covers a contiguous
time window of roughly 10–60 minutes.

In [ ]:
import sys
import gc
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import contextily as cx
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from IPython.display import display

sys.path.insert(0, str(Path('../../').resolve()))
from raw_sensor_integration.gps_parser import haversine_m

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path('.')
CSV_FILES = sorted(NOTEBOOK_DIR.glob('gps_2*.csv'))
SPEED_KNOTS_MAX = 15
MIN_DT_S = 0.5

# ---------------------------------------------------------------------------
# Load all CSVs into one DataFrame, keyed by .raw file
# ---------------------------------------------------------------------------
parts = []
for f in CSV_FILES:
    parts.append(pd.read_csv(f, parse_dates=['timestamp']))
raw_all = pd.concat(parts, ignore_index=True)

# Sort files chronologically (the filename encodes the start time)
file_order = sorted(raw_all['file'].unique())
raw_all['file'] = pd.Categorical(raw_all['file'], categories=file_order, ordered=True)
raw_all = raw_all.sort_values(['file', 'timestamp']).reset_index(drop=True)

print(f'Loaded {len(raw_all):,} rows across {len(file_order)} .raw files')

# ---------------------------------------------------------------------------
# Short label for each file (date + HH:MM)
# ---------------------------------------------------------------------------
def short_label(fname):
    # D20240131-T105410.raw -> 01-31 10:54
    stem = fname.replace('.raw', '')
    date_part = stem[1:9]   # 20240131
    time_part = stem[11:17] # 105410
    return f'{date_part[4:6]}-{date_part[6:8]} {time_part[0:2]}:{time_part[2:4]}'

file_labels = {f: short_label(f) for f in file_order}

# ---------------------------------------------------------------------------
# Deduplication: round to 1s, prefer GGA
# ---------------------------------------------------------------------------
TYPE_RANK = {'$GPGGA': 0, '$GNGGA': 0, '$INGGA': 0,
             '$GPRMC': 1, '$GNRMC': 1,
             '$GPGLL': 2}

deduped = {}
for fname in file_order:
    df = raw_all[raw_all['file'] == fname].copy()
    df['ts_round'] = df['timestamp'].dt.round('1s')
    df['type_rank'] = df['sentence_type'].map(TYPE_RANK).fillna(9)
    df = (df.sort_values(['ts_round', 'type_rank'])
            .drop_duplicates(subset='ts_round')
            .sort_values('timestamp')
            .reset_index(drop=True))
    deduped[fname] = df

# ---------------------------------------------------------------------------
# Compute per-file stats
# ---------------------------------------------------------------------------
unique_pos = {f: len(d.drop_duplicates(subset=['lat', 'lon'])) for f, d in deduped.items()}

# ---------------------------------------------------------------------------
# Compute speeds (skip frozen files)
# ---------------------------------------------------------------------------
speed_data = {}
for fname, d in deduped.items():
    if unique_pos[fname] < 10:
        continue
    g = d.sort_values('timestamp').copy()
    g['dt_s'] = g['timestamp'].diff().dt.total_seconds()
    dists = [float('nan')]
    for j in range(1, len(g)):
        dists.append(haversine_m(
            g['lat'].iloc[j-1], g['lon'].iloc[j-1],
            g['lat'].iloc[j],   g['lon'].iloc[j]))
    g['dist_m'] = dists
    g['speed_kn'] = (g['dist_m'] / g['dt_s']) * (3600 / 1852)
    g.loc[g['dt_s'] < MIN_DT_S, ['dist_m', 'speed_kn']] = float('nan')
    speed_data[fname] = g

print(f'Deduped files: {len(deduped)}, speed-computed: {len(speed_data)}')

## Overview

In [ ]:
rows = []
for fname in file_order:
    d = deduped[fname]
    dur_min = (d['timestamp'].max() - d['timestamp'].min()).total_seconds() / 60
    rows.append({
        'file': fname,
        'label': file_labels[fname],
        'raw_rows': len(raw_all[raw_all['file'] == fname]),
        'deduped': len(d),
        'duration_min': round(dur_min, 1),
        'unique_pos': unique_pos[fname],
        'start': d['timestamp'].min(),
    })

overview = pd.DataFrame(rows).set_index('file')
with pd.option_context('display.max_rows', 100):
    display(overview)
print(f'\nTotal raw rows: {overview["raw_rows"].sum():,}  |  '
      f'Total deduped: {overview["deduped"].sum():,}  |  '
      f'Files: {len(overview)}')

## Recording Timeline

In [ ]:
fig, ax = plt.subplots(figsize=(14, 25), dpi=130)

cmap = plt.cm.viridis
n = len(file_order)

for i, fname in enumerate(file_order):
    d = deduped[fname]
    t0 = d['timestamp'].min()
    t1 = d['timestamp'].max()
    is_frozen = unique_pos[fname] < 10
    c = '#d32f2f' if is_frozen else cmap(i / max(n - 1, 1))
    ax.barh(i, t1 - t0, left=t0, height=0.7, color=c, alpha=0.85,
            edgecolor='white', linewidth=0.3)

ax.set_yticks(range(n))
ax.set_yticklabels([file_labels[f] for f in file_order], fontsize=7)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
ax.set_xlabel('Time (UTC)')
ax.set_title('Recording Timeline — Per .raw File  (red = frozen GPS)')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

## Frozen GPS Detection

In [ ]:
frozen_stats = []
for fname in file_order:
    d = deduped[fname]
    same = ((d['lat'].diff().abs() < 1e-8) & (d['lon'].diff().abs() < 1e-8)).sum()
    frozen_pct = 100 * same / max(len(d) - 1, 1)
    frozen_stats.append({
        'file': fname,
        'unique_pos': unique_pos[fname],
        'frozen_%': round(frozen_pct, 1),
    })

frozen_df = pd.DataFrame(frozen_stats)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 25), dpi=130)

y = np.arange(len(file_order))
ylabels = [file_labels[f] for f in file_order]

# Left: unique positions
colors1 = ['#d32f2f' if v < 10 else 'steelblue' for v in frozen_df['unique_pos']]
ax1.barh(y, frozen_df['unique_pos'], color=colors1, edgecolor='white', linewidth=0.3, height=0.7)
ax1.set_yticks(y)
ax1.set_yticklabels(ylabels, fontsize=7)
ax1.set_xlabel('Unique (lat, lon) pairs')
ax1.set_title('Spatial Diversity')
ax1.grid(axis='x', alpha=0.3)
ax1.invert_yaxis()

# Right: frozen %
colors2 = ['#d32f2f' if v > 95 else '#ff9800' if v > 50 else '#4CAF50'
           for v in frozen_df['frozen_%']]
ax2.barh(y, frozen_df['frozen_%'], color=colors2, edgecolor='white', linewidth=0.3, height=0.7)
ax2.set_yticks(y)
ax2.set_yticklabels([], fontsize=7)
ax2.set_xlabel('Consecutive identical positions (%)')
ax2.set_title('Frozen GPS Indicator')
ax2.set_xlim(0, 110)
ax2.axvline(50, color='orange', linestyle='--', alpha=0.5)
ax2.axvline(95, color='red', linestyle='--', alpha=0.5)
ax2.grid(axis='x', alpha=0.3)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

frozen_files = [f for f in file_order if unique_pos[f] < 10]
print(f'Frozen files ({len(frozen_files)}):')
for f in frozen_files:
    print(f'  {f}  ({file_labels[f]})')

## GPS Tracks — Colored by File

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9), dpi=130)

cmap = plt.cm.hsv
n_active = sum(1 for f in file_order if unique_pos[f] >= 10)
active_idx = 0

for fname in file_order:
    d = deduped[fname]
    if unique_pos[fname] < 10:
        ax.plot(d['lon'].iloc[0], d['lat'].iloc[0], 'X', color='red',
                markersize=14, markeredgecolor='darkred', markeredgewidth=1.5,
                zorder=10)
    else:
        c = cmap(active_idx / max(n_active - 1, 1))
        ax.plot(d['lon'], d['lat'], '-', color=c, linewidth=0.8, alpha=0.7, zorder=3)
        active_idx += 1

cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.CartoDB.Positron,
               zoom=13, attribution_size=6)

# Colorbar representing chronological order
sm = ScalarMappable(cmap=cmap, norm=Normalize(vmin=0, vmax=n_active - 1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.01)
# Label colorbar with first/last file times
active_files = [f for f in file_order if unique_pos[f] >= 10]
cbar.set_ticks([0, n_active - 1])
cbar.set_ticklabels([file_labels[active_files[0]], file_labels[active_files[-1]]])
cbar.set_label('Recording time (earliest → latest)', fontsize=9)

ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('GPS Tracks — Colored by .raw File  (red X = frozen)')
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

## Speed Map

In [ ]:
all_valid = pd.concat([d.dropna(subset=['speed_kn']) for d in speed_data.values()])

fig, ax = plt.subplots(figsize=(12, 9), dpi=130)

norm = Normalize(vmin=0, vmax=all_valid['speed_kn'].quantile(0.95))
good = all_valid[all_valid['speed_kn'] <= SPEED_KNOTS_MAX]
bad  = all_valid[all_valid['speed_kn'] > SPEED_KNOTS_MAX]

ax.scatter(good['lon'], good['lat'], c=good['speed_kn'], cmap='plasma',
           norm=norm, s=3, linewidths=0, zorder=3, alpha=0.7)

if len(bad) > 0:
    ax.scatter(bad['lon'], bad['lat'], s=40, facecolors='none',
               edgecolors='red', linewidths=1.5, zorder=5, alpha=0.8,
               label=f'Speed spikes > {SPEED_KNOTS_MAX} kn (n={len(bad)})')

cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.CartoDB.Positron,
               zoom=13, attribution_size=6)

cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='plasma'), ax=ax,
                    fraction=0.025, pad=0.01)
cbar.set_label('Speed (knots)', fontsize=9)

ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('GPS Track Coloured by Speed — Red Circles = Anomalous Spikes')
ax.legend(fontsize=9, loc='upper left', framealpha=0.9)
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

## Quality Scorecard

In [ ]:
score_rows = []
for fname in file_order:
    d = deduped[fname]
    n_unique = unique_pos[fname]
    dur_min = (d['timestamp'].max() - d['timestamp'].min()).total_seconds() / 60

    # Frozen %
    same = ((d['lat'].diff().abs() < 1e-8) & (d['lon'].diff().abs() < 1e-8)).sum()
    fz = 100 * same / max(len(d) - 1, 1)

    # Temporal: median dt
    dt = d['timestamp'].diff().dt.total_seconds().dropna()
    med_dt = dt.median() if len(dt) > 0 else float('nan')
    gaps_5s = int((dt > 5).sum()) if len(dt) > 0 else 0

    if fname in speed_data:
        sd = speed_data[fname]
        v  = sd['speed_kn'].dropna()
        spikes_15 = int((v > SPEED_KNOTS_MAX).sum())
        max_kn = round(v.max(), 1) if len(v) > 0 else 0
        total_km = round(sd['dist_m'].dropna().sum() / 1000, 2)
        med_kn = round(v.median(), 2) if len(v) > 0 else 0
    else:
        spikes_15 = 0
        max_kn = 0
        total_km = 0
        med_kn = 0

    # Grade
    if n_unique < 10:
        grade = 'F'
    elif spikes_15 > 50:
        grade = 'C'
    elif spikes_15 > 10:
        grade = 'B'
    else:
        grade = 'A'

    score_rows.append({
        'file': fname,
        'label': file_labels[fname],
        'unique_pos': n_unique,
        'frozen_%': round(fz, 1),
        'duration_min': round(dur_min, 1),
        'median_kn': med_kn,
        'max_kn': max_kn,
        'spikes_>15kn': spikes_15,
        'total_km': total_km,
        'grade': grade,
    })

scorecard = pd.DataFrame(score_rows).set_index('file')

In [ ]:
# ---------------------------------------------------------------------------
# Quality heatmap
# ---------------------------------------------------------------------------
metrics = ['unique_pos', 'frozen_%', 'spikes_>15kn', 'max_kn', 'total_km', 'median_kn']
hm = scorecard[metrics].copy()

# Normalize: 1 = best, 0 = worst
hm_norm = hm.copy()
for col in ['unique_pos', 'total_km', 'median_kn']:
    r = hm[col].max() - hm[col].min()
    hm_norm[col] = (hm[col] - hm[col].min()) / r if r > 0 else 0.5
for col in ['frozen_%', 'spikes_>15kn', 'max_kn']:
    r = hm[col].max() - hm[col].min()
    hm_norm[col] = 1 - (hm[col] - hm[col].min()) / r if r > 0 else 0.5

fig, ax = plt.subplots(figsize=(14, 28), dpi=130)
im = ax.imshow(hm_norm.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

# Annotate with raw values
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        val = hm.iloc[i, j]
        fmt = f'{val:,.0f}' if val > 10 else f'{val:.1f}'
        text_color = 'white' if hm_norm.iloc[i, j] < 0.3 else 'black'
        ax.text(j, i, fmt, ha='center', va='center', fontsize=6,
                fontweight='bold', color=text_color)

ax.set_xticks(range(len(metrics)))
ax.set_xticklabels(['Unique\npositions', 'Frozen %', 'Spikes\n> 15 kn',
                     'Max speed\n(kn)', 'Total dist\n(km)', 'Median speed\n(kn)'],
                    fontsize=9)
ax.set_yticks(range(len(scorecard)))
ax.set_yticklabels([file_labels[f] for f in scorecard.index], fontsize=6)

# Grade labels on right side
grade_colors = {'A': '#4CAF50', 'B': '#FF9800', 'C': '#f44336', 'F': '#b71c1c'}
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
ax2.set_yticks(range(len(scorecard)))
ax2.set_yticklabels([f"  {g}" for g in scorecard['grade']],
                     fontsize=8, fontweight='bold')
for i, g in enumerate(scorecard['grade']):
    ax2.get_yticklabels()[i].set_color(grade_colors.get(g, 'black'))

cbar = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.08)
cbar.set_label('Quality (green = good, red = bad)', fontsize=9)

ax.set_title('GPS Data Quality — Per .raw File Scorecard', fontsize=13)
plt.tight_layout()
plt.show()
plt.close(fig)
gc.collect()

# Grade summary
grade_counts = scorecard['grade'].value_counts().sort_index()
print('Grade distribution:')
for g, c in grade_counts.items():
    print(f'  {g}: {c} files')

## Tiered Echogram Spot-Check (A, B, C, F)

Use the existing raw loading and echogram visualization utilities to inspect representative files from each GPS quality tier.


In [ ]:
from raw_sensor_integration.data_loader import load_raw
from raw_sensor_integration.visualization import plot_echogram

# ---------------------------------------------------------------------------
# Resolve .raw paths from file names in scorecard
# ---------------------------------------------------------------------------
RAW_SEARCH_ROOTS = [
    Path('../../../Datasets/raw_files'),
    Path('/Users/axelhagen/Master/Datasets/raw_files'),
]

MAX_PINGS_ECHOGRAM = 900
TIER_N_EXAMPLES = {'A': 2, 'B': 2, 'C': 2, 'F': 2}


def resolve_raw_file(fname: str):
    for root in RAW_SEARCH_ROOTS:
        if not root.exists():
            continue
        matches = list(root.rglob(fname))
        if matches:
            return matches[0]
    return None


def representative_rows(df, n):
    if len(df) <= n:
        return df
    idx = np.linspace(0, len(df) - 1, num=n, dtype=int)
    return df.iloc[idx]


scorecard_work = scorecard.reset_index().copy()
scorecard_work['start'] = scorecard_work['file'].map(lambda f: deduped[f]['timestamp'].min())
scorecard_work['raw_path'] = scorecard_work['file'].map(resolve_raw_file)
scorecard_work['raw_found'] = scorecard_work['raw_path'].notna()

print('Raw-file availability by grade:')
print(scorecard_work.groupby('grade')['raw_found'].agg(['sum', 'count']).rename(columns={'sum': 'found'}))

# Select representative files from each grade (chronological coverage)
tier_examples = []
for grade in ['A', 'B', 'C', 'F']:
    g = scorecard_work[(scorecard_work['grade'] == grade) & (scorecard_work['raw_found'])].sort_values('start')
    pick = representative_rows(g, TIER_N_EXAMPLES.get(grade, 2))
    tier_examples.append(pick)

examples_df = pd.concat(tier_examples, ignore_index=True) if tier_examples else pd.DataFrame()

if examples_df.empty:
    print('No tier examples could be resolved to .raw files.')
else:
    display(examples_df[['file', 'label', 'grade', 'unique_pos', 'frozen_%', 'spikes_>15kn', 'raw_path']])


In [ ]:
# ---------------------------------------------------------------------------
# Plot echograms for representative A/B/C/F files
# ---------------------------------------------------------------------------
if examples_df.empty:
    print('Skipping echogram plots because no raw files were found.')
else:
    for row in examples_df.itertuples(index=False):
        raw_path = Path(row.raw_path)
        try:
            ed = load_raw(raw_path, max_pings=MAX_PINGS_ECHOGRAM, include_gps=False)
            sv = np.clip(ed.sv_db, -100, 0)
            vmin, vmax = np.nanpercentile(sv, [2, 98])

            fig, ax = plot_echogram(
                sv,
                title=f"Tier {row.grade} | {row.file}",
                vmin=vmin,
                vmax=vmax,
                cmap='viridis',
                range_array=ed.range_array,
            )
            ax.set_title(
                f"Tier {row.grade} | {row.file} | {row.label} | "
                f"n={ed.n_pings} pings | ch≈{ed.frequency_hz/1000:.0f} kHz"
            )
            plt.tight_layout()
            plt.show()
            plt.close(fig)
            gc.collect()
        except Exception as e:
            print(f"[WARN] Could not plot {row.file}: {e}")


## Raw `S_v` Data Exploration and Outlier Detection

Compute robust per-file statistics directly from `.raw` echograms and detect outliers in both value distribution and ping-level dynamics.


In [ ]:
# ---------------------------------------------------------------------------
# Per-file S_v exploration (robust stats + outlier flags)
# ---------------------------------------------------------------------------
SV_MAX_PINGS_PER_FILE = 500
SV_ANALYSIS_MAX_FILES = None  # set e.g. 30 for a faster exploratory pass

available = scorecard_work[scorecard_work['raw_found']].copy().sort_values('start')
if SV_ANALYSIS_MAX_FILES is not None:
    available = available.head(SV_ANALYSIS_MAX_FILES)

sv_rows = []
sv_ping_series = {}

for row in available.itertuples(index=False):
    try:
        ed = load_raw(Path(row.raw_path), max_pings=SV_MAX_PINGS_PER_FILE, include_gps=False)
        sv = np.clip(ed.sv_db, -100, 0)

        vals = sv[np.isfinite(sv)]
        if vals.size < 1000:
            continue

        q01, q05, q25, q50, q75, q95, q99 = np.nanpercentile(vals, [1, 5, 25, 50, 75, 95, 99])
        iqr = q75 - q25

        lower = q25 - 3.0 * iqr
        upper = q75 + 3.0 * iqr
        frac_extreme = np.mean((vals < lower) | (vals > upper))
        frac_very_high = np.mean(vals > -20)
        frac_very_low = np.mean(vals < -95)

        ping_med = np.nanmedian(sv, axis=0)
        ping_p95 = np.nanpercentile(sv, 95, axis=0)

        pm_med = np.nanmedian(ping_med)
        pm_mad = np.nanmedian(np.abs(ping_med - pm_med))
        if pm_mad < 1e-6:
            ping_outlier_frac = 0.0
        else:
            robust_z = np.abs(ping_med - pm_med) / (1.4826 * pm_mad)
            ping_outlier_frac = np.mean(robust_z > 5.0)

        sv_rows.append({
            'file': row.file,
            'label': row.label,
            'grade': row.grade,
            'n_pings': ed.n_pings,
            'n_samples': ed.n_samples,
            'q01': q01,
            'q05': q05,
            'q25': q25,
            'q50': q50,
            'q75': q75,
            'q95': q95,
            'q99': q99,
            'iqr': iqr,
            'std': float(np.nanstd(vals)),
            'frac_extreme_iqr3': frac_extreme,
            'frac_gt_-20dB': frac_very_high,
            'frac_lt_-95dB': frac_very_low,
            'ping_outlier_frac': ping_outlier_frac,
            'raw_path': str(row.raw_path),
        })

        sv_ping_series[row.file] = {
            'ping_med': ping_med,
            'ping_p95': ping_p95,
            'range_array': ed.range_array,
        }

    except Exception as e:
        print(f"[WARN] S_v stats failed for {row.file}: {e}")

sv_stats = pd.DataFrame(sv_rows)

if sv_stats.empty:
    print('No S_v stats were computed (check raw paths and limits).')
else:
    # Composite outlier score via robust z on multiple indicators
    outlier_features = [
        'frac_extreme_iqr3',
        'frac_gt_-20dB',
        'frac_lt_-95dB',
        'std',
        'ping_outlier_frac',
    ]

    for col in outlier_features:
        med = sv_stats[col].median()
        mad = np.median(np.abs(sv_stats[col] - med))
        if mad < 1e-12:
            sv_stats[f'rz_{col}'] = 0.0
        else:
            sv_stats[f'rz_{col}'] = np.abs(sv_stats[col] - med) / (1.4826 * mad)

    rz_cols = [f'rz_{c}' for c in outlier_features]
    sv_stats['outlier_score'] = sv_stats[rz_cols].mean(axis=1)
    sv_stats = sv_stats.sort_values('outlier_score', ascending=False).reset_index(drop=True)

    print(f"Computed S_v stats for {len(sv_stats)} files")
    display(sv_stats[['file', 'grade', 'n_pings', 'q01', 'q50', 'q99',
                      'frac_extreme_iqr3', 'ping_outlier_frac', 'outlier_score']].head(20))


In [ ]:
# ---------------------------------------------------------------------------
# S_v outlier plots
# ---------------------------------------------------------------------------
if sv_stats.empty:
    print('Skipping S_v plots: no stats available.')
else:
    grade_palette = {'A': '#4CAF50', 'B': '#FF9800', 'C': '#f44336', 'F': '#6A1B9A'}
    c = sv_stats['grade'].map(grade_palette).fillna('gray')

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=130)

    axes[0, 0].scatter(sv_stats['q99'], sv_stats['q01'], c=c, s=40, alpha=0.8)
    axes[0, 0].set_xlabel('q99 (dB)')
    axes[0, 0].set_ylabel('q01 (dB)')
    axes[0, 0].set_title('Distribution Tails by File')
    axes[0, 0].grid(alpha=0.25)

    axes[0, 1].hist(sv_stats['outlier_score'], bins=20, color='slateblue', alpha=0.85)
    axes[0, 1].set_xlabel('Composite outlier score')
    axes[0, 1].set_ylabel('Number of files')
    axes[0, 1].set_title('Outlier Score Distribution')
    axes[0, 1].grid(alpha=0.25)

    topn = min(15, len(sv_stats))
    top = sv_stats.head(topn).iloc[::-1]
    axes[1, 0].barh(top['file'], top['outlier_score'], color='firebrick', alpha=0.9)
    axes[1, 0].set_xlabel('Outlier score')
    axes[1, 0].set_title(f'Top {topn} S_v Outlier Files')
    axes[1, 0].tick_params(axis='y', labelsize=7)
    axes[1, 0].grid(axis='x', alpha=0.25)

    axes[1, 1].scatter(sv_stats['ping_outlier_frac'], sv_stats['frac_extreme_iqr3'], c=c, s=40, alpha=0.8)
    axes[1, 1].set_xlabel('Ping-level outlier fraction')
    axes[1, 1].set_ylabel('Extreme-value fraction (IQR*3 rule)')
    axes[1, 1].set_title('Temporal vs Value Outliers')
    axes[1, 1].grid(alpha=0.25)

    for grade, color in grade_palette.items():
        axes[0, 0].scatter([], [], c=color, label=grade)
    axes[0, 0].legend(title='GPS grade', fontsize=8)

    plt.tight_layout()
    plt.show()
    plt.close(fig)
    gc.collect()


In [ ]:
# ---------------------------------------------------------------------------
# Inspect top S_v outliers with echogram + ping-median trace
# ---------------------------------------------------------------------------
SV_TOP_OUTLIERS_TO_PLOT = 6

if sv_stats.empty:
    print('Skipping outlier echogram inspection: no stats available.')
else:
    top_files = sv_stats.head(SV_TOP_OUTLIERS_TO_PLOT)

    for row in top_files.itertuples(index=False):
        try:
            ed = load_raw(Path(row.raw_path), max_pings=SV_MAX_PINGS_PER_FILE, include_gps=False)
            sv = np.clip(ed.sv_db, -100, 0)
            vmin, vmax = np.nanpercentile(sv, [2, 98])

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), dpi=130,
                                           gridspec_kw={'height_ratios': [3, 1]},
                                           sharex=True)

            # Reuse existing echogram plotting utility on provided axis
            plot_echogram(
                sv,
                title=f"Top S_v outlier | {row.file} | score={row.outlier_score:.2f}",
                vmin=vmin,
                vmax=vmax,
                cmap='viridis',
                range_array=ed.range_array,
                ax=ax1,
            )

            ping_med = np.nanmedian(sv, axis=0)
            pm_med = np.nanmedian(ping_med)
            pm_mad = np.nanmedian(np.abs(ping_med - pm_med))
            if pm_mad < 1e-6:
                outlier_mask = np.zeros_like(ping_med, dtype=bool)
            else:
                robust_z = np.abs(ping_med - pm_med) / (1.4826 * pm_mad)
                outlier_mask = robust_z > 5.0

            x = np.arange(len(ping_med))
            ax2.plot(x, ping_med, color='black', linewidth=1.0, label='Ping median S_v (dB)')
            if outlier_mask.any():
                ax2.scatter(x[outlier_mask], ping_med[outlier_mask], color='red', s=14,
                            label=f'Outlier pings (n={outlier_mask.sum()})')
            ax2.axhline(pm_med, color='steelblue', linestyle='--', linewidth=1.0,
                        label=f'Median={pm_med:.1f} dB')

            ax2.set_xlabel('Ping number')
            ax2.set_ylabel('Median dB')
            ax2.grid(alpha=0.25)
            ax2.legend(loc='best', fontsize=8)
            ax2.set_title(f"Ping-level dynamics | grade={row.grade} | q01={row.q01:.1f}, q99={row.q99:.1f}")

            plt.tight_layout()
            plt.show()
            plt.close(fig)
            gc.collect()

        except Exception as e:
            print(f"[WARN] Outlier inspection failed for {row.file}: {e}")
